# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
from IPython.display import Markdown, display
from openai import OpenAI
import os
from dotenv import load_dotenv

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
MODEL_GEMINI = 'gemini-3.1-flash-lite'

In [3]:
# set up environment
load_dotenv(override=True)


# Load environment variables in a file called .env
api_key = os.getenv('GOOGLE_API_KEY')

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OLLAMA_BASE_URL = "http://localhost:11434/v1"


if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("AQ."):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

API key found and looks good so far!


In [4]:
system_prompt = """
    You are an helpful assistant to help understand the code and the concepts behind it and suggest the best way to implement it.
    You are also an expert in the field of software development and have a deep understanding of the code and the concepts behind it.
"""

In [5]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:

def fetch_website_contents(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]
"""

In [6]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

In [ ]:
# Get gpt-4o-mini to answer, with streaming

In [8]:
# Get Llama 3.2 to answer
# !ollama pull llama3.2

# Step 3: Call OpenAI
response = ollama.chat.completions.create(
    model=MODEL_LLAMA, 
    messages=messages
)

# Step 4: print the result
Markdown(response.choices[0].message.content)

**Explanation of the Code**

This code defines a function called `fetch_website_contents` which takes a URL as input and returns a limited portion of the website's contents.

Here's a step-by-step breakdown:

1. **Make a GET Request**: The function uses the `requests` library to send a GET request to the provided URL with custom headers.

2. **Parse HTML Content**: The response content is then passed to `BeautifulSoup`, a HTML and XML parser, to parse the HTML content of the webpage.

3. **Extract Page Title**: The title of the webpage is extracted using the `title` attribute of the parsed HTML.

4. **Remove Irrelevant Elements**: The code then iterates through the `body` elements of the parsed HTML (excluding any elements identified by `["script", "style", "img", "input"]`) and removes them using the `decompose` method. This process creates a cleaner representation of the webpage's content.

5. **Extract Text Content**: The code then extracts the text content of the webpage using the `get_text` method on the `body` element. It also sets a separator to use when joining multiple paragraphs of text and strips any whitespace from the extracted text.

6. **Limit the Content**: The extracted title and text content are then combined and the result is truncated to 2000 characters (replacing any longer content with "[...]")
 

7. **Return the Result**: The function finally returns the truncated content.

**Possible Issues and Considerations**

- **User Aggreement and Security**: Be mindful of website user agreements and terms of use when accessing and parsing other websites' content (for example, public websites may prohibit scraping).

- **Content Limit**

- **Inconsistency of Content Format**: If multiple types of content (such as image or video content) are combined together the code is unable to parse these correctly.

- **Data Accuracy**

- **Over-fetching**

- **Security**

- **Performance**: Consider optimizing the code for performance if the function needs to be called frequently. Also if you expect this function to return for long-running web content then consider improving performance.  

- **Code Structure**: This function appears to perform multiple operations that can be broken down into separate functions with more descriptive names, e.g. `connect_to_url`, `parse_html`, `extract_text`, `format_content`.

In [9]:
# Step 3: Call OpenAI
response = gemini.chat.completions.create(
    model=MODEL_GEMINI, 
    messages=messages
)

# Step 4: print the result
Markdown(response.choices[0].message.content)

### Explanation of the Code

This function, `fetch_website_contents`, is designed to **extract and clean the primary text content** from a given URL. It is commonly used in web scraping, data gathering for LLMs (Large Language Models), or content summarization tasks.

Here is a step-by-step breakdown of what it does:

1.  **Requesting the Page:** It uses `requests.get()` to download the HTML source code of the website. It includes a `headers` object (not shown in your snippet, but implied) to mimic a real browser, which helps avoid being blocked by anti-bot measures.
2.  **Parsing:** It uses `BeautifulSoup` to turn the raw HTML string into a navigable tree structure.
3.  **Extracting the Title:** It looks for the `<title>` tag. If it exists, it keeps the text; otherwise, it assigns a default string.
4.  **Cleaning (Decomposition):** This is the most critical part. Before extracting the text, it searches within the `<body>` for specific tags that usually contain "noise" rather than meaningful information:
    *   `<script>` and `<style>`: These contain code and CSS, not human-readable text.
    *   `<img>` and `<input>`: These represent visual elements or interactive forms that don't contribute to the "article" content.
    *   `.decompose()`: This method **removes these tags entirely** from the HTML tree.
5.  **Extracting Text:** It converts the remaining HTML structure into plain text. The `separator="\n"` argument adds line breaks between blocks to maintain readability, and `strip=True` removes unnecessary whitespace.
6.  **Truncating:** Finally, it limits the result to the first **2,000 characters**.

---

### Why is this code written this way?

*   **Noise Reduction:** If you don't remove `<script>` and `<style>` tags, their internal code (JavaScript logic and CSS rules) would be included in your output, making the data unusable for analysis or language modeling.
*   **Token Efficiency:** By truncating the output to 2,000 characters, the developer is likely preparing the data for an LLM (like GPT). LLMs have "context windows," and stripping out irrelevant HTML elements ensures that you use your token budget for actual content rather than boilerplate code.
*   **Structure Preservation:** Using `separator="\n"` ensures that headers, paragraphs, and list items don't just collapse into one giant, unreadable block of text.

---

### Expert Suggestions for Improvement

While this code works for simple sites, it has a few weaknesses you should consider for production:

#### 1. Robustness (Error Handling)
The current code will crash if the URL is invalid or the connection times out.
*   **Recommendation:** Wrap the `requests.get()` call in a `try...except` block and check `response.raise_for_status()` to catch HTTP errors (like 404 or 500).

#### 2. Performance
Parsing HTML with `BeautifulSoup` is relatively slow. 
*   **Recommendation:** If you are scraping thousands of pages, consider using `lxml` as the parser for BeautifulSoup instead of `html.parser`. It is significantly faster.
    ```python
    soup = BeautifulSoup(response.content, "lxml")
    ```

#### 3. Modern Web Rendering
Many modern websites (like React or Vue apps) load their content using JavaScript. `requests` only downloads the initial HTML; it does not execute JavaScript.
*   **Recommendation:** If the page returns "No title" or empty text, you likely need a browser automation tool like **Playwright** or **Selenium** to render the JavaScript before extracting the content.

#### 4. Semantic Cleaning
Removing `img` and `input` is good, but you might also want to remove `<nav>`, `<footer>`, and `<aside>` tags, as these contain navigation links and footers that are almost always irrelevant to the main article.
*   **Recommendation:** Add these to your `decompose` list:
    ```python
    for tag in soup.body(["script", "style", "nav", "footer", "aside"]):
        tag.decompose()
    ```

#### 5. Character Limit
Truncating by characters (`[:2000]`) can cut off a word in the middle. 
*   **Recommendation:** If you are using this for LLMs, consider using a proper tokenizer (like `tiktoken` for OpenAI models) to measure length, as token count is more accurate than character count.